# Agentic Financial Intelligence — Interactive Demo

**Nebius × NVIDIA Global AI Hackathon**

Explore an agentic financial intelligence prototype combining quantitative market analytics and FY2025 corporate financial documentation for 30 Dow Jones Industrial Average companies. The workflow uses **LangGraph** for orchestration, **NVIDIA Nemotron via Nebius Token Factory** for AI reasoning, and **Gradio** for the interactive interface.

### Before you start

You will need your **own two API keys**:

1. **Nebius Token Factory:** add `NEBIUS_API_KEY` in **Google Colab → Secrets** (key icon on the left).
2. **Twelve Data:** add `TWELVE_DATA` in the same **Secrets** panel.

For **both** secrets, turn on **Notebook access**. Do not paste keys into the notebook's code or share them publicly. **LangGraph and Gradio do not require additional API keys.**

### Start the demo — one click

1. Connect to a Colab runtime if prompted.
2. Click the **▶ play button** on the single collapsed **Launch Interactive Demo** cell below. **You do not need to expand or edit the code.**
3. **Please allow a few minutes** for dependencies to install, the public GitHub project modules to load, and the Gradio landing page to appear below the cell.
4. Once the interface is visible, enter your financial question and select **Analyze**.

The notebook downloads the public project from GitHub and runs it **in your own Colab runtime**. It does not use the author's Colab session or API credentials. If the session is restarted, click ▶ again.


In [1]:
# 1 — Load the original project and install its dependencies.
import os
import sys
import subprocess
from pathlib import Path

ROOT = Path('/content/agentic-financial-intelligence')
REPO = 'https://github.com/Valeria-Raele/agentic-financial-intelligence.git'

if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)

assert (ROOT / 'gradio_app.py').is_file(), 'gradio_app.py missing'
assert (ROOT / 'hf_engine.py').is_file(), 'hf_engine.py missing'
requirements = ROOT / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError('requirements.txt missing from the repository')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
print('Project modules loaded.')


# 2 — Read API credentials from Colab Secrets (never print their values).
from google.colab import userdata

for secret, aliases in (
    ('NEBIUS_API_KEY', ('NEBIUS_API_KEY',)),
    ('TWELVE_DATA', ('TWELVE_DATA', 'TWELVE_DATA_API_KEY')),
):
    try:
        value = userdata.get(secret)
    except Exception:
        value = None
    if not value:
        raise RuntimeError(f'Missing Colab Secret: {secret}. Add it in the Secrets panel and enable notebook access.')
    for alias in aliases:
        os.environ[alias] = value
print('API credentials verified.')


# 3 — Launch the original Gradio app in Colab (no diagnostics or engine changes).
import ast
import re
import time
import socket

app_file = ROOT / 'gradio_app.py'
original = subprocess.check_output(['git', 'show', 'HEAD:gradio_app.py'], cwd=ROOT, text=True)

# Remove only Hugging Face ZeroGPU decorators/imports that are not used in Colab.
cleaned = []
for line in original.splitlines(keepends=True):
    stripped = line.strip()
    if re.fullmatch(r'import\s+spaces\s*(?:#.*)?', stripped):
        continue
    if re.fullmatch(r'@spaces\.GPU(?:\s*\(.*\))?\s*(?:#.*)?', stripped):
        continue
    cleaned.append(line)
updated = ''.join(cleaned)

# Keep the existing Gradio UI, callback, engine integration and module logic.
for required in ('def run_question(', 'initialize_engine_once', 'from hf_engine import ask', 'with gr.Blocks('):
    if required not in updated:
        raise RuntimeError(f'Unexpected gradio_app.py structure: {required} not found')

launch_marker = 'demo.queue(default_concurrency_limit=1).launch('
if launch_marker not in updated:
    raise RuntimeError('Original Gradio launch statement not recognized')
if 'server_name="0.0.0.0"' not in updated:
    updated = updated.replace(launch_marker, launch_marker + 'server_name="0.0.0.0", server_port=7860, ', 1)

# Fail explicitly if a different kind of spaces dependency remains.
tree = ast.parse(updated, filename=str(app_file))
for node in ast.walk(tree):
    if isinstance(node, ast.ImportFrom) and node.module and node.module.startswith('spaces'):
        raise RuntimeError('Unsupported spaces import in app')
    if isinstance(node, ast.Attribute):
        root = node
        while isinstance(root, ast.Attribute):
            root = root.value
        if isinstance(root, ast.Name) and root.id == 'spaces':
            raise RuntimeError('Unsupported spaces reference in app')
app_file.write_text(updated, encoding='utf-8')

# Launch the real app.
if 'demo_process' in globals() and demo_process.poll() is None:
    demo_process.terminate()
    demo_process.wait(timeout=10)

log_path = ROOT / '_demo_startup.log'
with log_path.open('w', encoding='utf-8') as log:
    demo_process = subprocess.Popen(
        [sys.executable, '-u', str(app_file)], cwd=str(ROOT),
        env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT
    )

for _ in range(90):
    if demo_process.poll() is not None:
        raise RuntimeError('Gradio startup failed:\n' + log_path.read_text(errors='replace')[-5000:])
    with socket.socket() as sock:
        sock.settimeout(1)
        if sock.connect_ex(('127.0.0.1', 7860)) == 0:
            break
    time.sleep(2)
else:
    raise TimeoutError('Gradio did not start within 180 seconds. Check startup log.')
print('Demo ready — open the interface using the link below.')


# 4 — Interactive financial intelligence demo (external browser tab only).
from google.colab import output
from IPython.display import display, HTML

url = output.eval_js("google.colab.kernel.proxyPort(7860)")
display(HTML(f'<a href="{url}" target="_blank" rel="noopener noreferrer">Apri Gradio in una nuova scheda</a>'))

Project modules loaded.
API credentials verified.
Demo ready — open the interface using the link below.
